# Phase 2 — Data Engineering
## Predicting Return Probability, Return Fraud & Product Quality Issues
### NMIMS MBA — Big Data Analytics Group Project

**Purpose of this notebook:** Take the raw e-commerce returns dataset (standing in for the Bronze-layer
landing zone that, in production, would be populated by Sqoop/Flume/API ingestion — see the
Enterprise Architecture section of the report) and produce a clean, feature-engineered
**Silver → Gold** dataset ready for PySpark MLlib modeling in Phase 3.

**Medallion mapping used in this notebook:**
- **Bronze** — raw CSV loaded as-is with an explicit schema (Section 1)
- **Silver** — validated, deduplicated, outlier-treated, feature-engineered table (Sections 2-4)
- **Gold** — encoded, scaled, model-ready feature vectors, split into train/test and persisted as Parquet (Section 5)

All decisions are justified inline — this notebook is graded on **Feature Engineering (3 marks)**
and feeds directly into **ML Implementation (4 marks)**.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import os

spark = (
    SparkSession.builder
    .appName("EcommerceReturnAbuse-Phase2-DataEngineering")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")   # small local dataset -> fewer shuffle partitions than the 200 default
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

26/09/11 10:53:20 WARN Utils: Your hostname, Yatharths-MacBook-Air-5.local resolves to a loopback address: 127.0.0.1; using 10.200.75.100 instead (on interface en0)
26/09/11 10:53:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/11 10:53:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Bronze Layer — Ingest with an explicit schema

In production this table is populated incrementally: Sqoop pulls the transactional `orders`/`returns`
tables nightly (`--incremental append --check-column order_date`), Flume tails the clickstream/event
logs for browsing behaviour (e.g. `wishlist_to_cart_time_hrs`), and a scheduled API job pulls
carrier tracking status. Here we simulate the result of that ingestion as a single Bronze CSV drop
and load it with an **explicit schema** (never `inferSchema` in production — it forces a full extra
data scan and can silently mis-type columns on schema drift).

In [ ]:
RAW_PATH = "../data/ecommerce_return_abuse_dataset.csv"

schema = T.StructType([
    T.StructField("order_id", T.StringType(), False),
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("age", T.IntegerType(), True),
    T.StructField("account_age_days", T.IntegerType(), True),
    T.StructField("customer_segment", T.StringType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("platform", T.StringType(), True),
    T.StructField("device_type", T.StringType(), True),
    T.StructField("payment_method", T.StringType(), True),
    T.StructField("product_category", T.StringType(), True),
    T.StructField("avg_order_value_usd", T.DoubleType(), True),
    T.StructField("refund_amount_requested_usd", T.DoubleType(), True),
    T.StructField("is_high_value_item", T.IntegerType(), True),
    T.StructField("discount_used", T.IntegerType(), True),
    T.StructField("order_date", T.DateType(), True),
    T.StructField("return_date", T.DateType(), True),
    T.StructField("days_to_return", T.IntegerType(), True),
    T.StructField("return_reason", T.StringType(), True),
    T.StructField("total_orders_lifetime", T.IntegerType(), True),
    T.StructField("total_returns_lifetime", T.IntegerType(), True),
    T.StructField("return_rate_pct", T.DoubleType(), True),
    T.StructField("item_returned_opened", T.IntegerType(), True),
    T.StructField("return_packaging_intact", T.IntegerType(), True),
    T.StructField("photo_evidence_provided", T.IntegerType(), True),
    T.StructField("tracking_number_valid", T.IntegerType(), True),
    T.StructField("shipping_carrier", T.StringType(), True),
    T.StructField("address_change_before_delivery", T.IntegerType(), True),
    T.StructField("refund_to_different_account", T.IntegerType(), True),
    T.StructField("multiple_accounts_flag", T.IntegerType(), True),
    T.StructField("customer_support_contacts", T.IntegerType(), True),
    T.StructField("previous_dispute_count", T.IntegerType(), True),
    T.StructField("wishlist_to_cart_time_hrs", T.DoubleType(), True),
    T.StructField("review_left_after_return", T.IntegerType(), True),
    T.StructField("abuse_type", T.StringType(), True),
    T.StructField("abuse_label", T.IntegerType(), True),
])

bronze_df = spark.read.csv(RAW_PATH, header=True, schema=schema)
bronze_df.cache()
print(f"Rows: {bronze_df.count():,}  |  Columns: {len(bronze_df.columns)}")
bronze_df.printSchema()

## 2. Data Quality Validation

The handoff notes this dataset was already verified clean (0 missing values, no duplicate
`order_id`). We re-verify programmatically here so the notebook is self-contained and reproducible
for the grader / any teammate re-running it, rather than taking that on faith.

In [3]:
# 2.1 Null audit across every column
null_counts = bronze_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in bronze_df.columns
])
null_counts.show(vertical=True, truncate=False)

-RECORD 0-----------------------------
 order_id                       | 0   
 customer_id                    | 0   
 age                            | 0   
 account_age_days               | 0   
 customer_segment               | 0   
 country                        | 0   
 platform                       | 0   
 device_type                    | 0   
 payment_method                 | 0   
 product_category               | 0   
 avg_order_value_usd            | 0   
 refund_amount_requested_usd    | 0   
 is_high_value_item             | 0   
 discount_used                  | 0   
 order_date                     | 0   
 return_date                    | 0   
 days_to_return                 | 0   
 return_reason                  | 0   
 total_orders_lifetime          | 0   
 total_returns_lifetime         | 0   
 return_rate_pct                | 0   
 item_returned_opened           | 0   
 return_packaging_intact        | 0   
 photo_evidence_provided        | 0   
 tracking_number_valid   

In [4]:
# 2.2 Duplicate order_id check
dup_orders = bronze_df.groupBy("order_id").count().filter("count > 1")
print(f"Duplicate order_id rows: {dup_orders.count()}")

# 2.3 Target class balance (confirms the 70/12/10/8 imbalance noted in the handoff)
bronze_df.groupBy("abuse_type", "abuse_label").count() \
    .withColumn("pct", F.round(F.col("count") / bronze_df.count() * 100, 1)) \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

Duplicate order_id rows: 0


+-----------------+-----------+-----+----+
|abuse_type       |abuse_label|count|pct |
+-----------------+-----------+-----+----+
|Legitimate       |0          |42060|70.1|
|Policy Abuser    |1          |7192 |12.0|
|Fraudulent Return|2          |6112 |10.2|
|Wardrobing       |3          |4636 |7.7 |
+-----------------+-----------+-----+----+



In [5]:
# 2.4 Referential/logical sanity checks
sanity = bronze_df.select(
    F.sum((F.col("return_date") < F.col("order_date")).cast("int")).alias("returns_before_order"),
    F.sum((F.col("total_returns_lifetime") > F.col("total_orders_lifetime")).cast("int")).alias("more_returns_than_orders"),
    F.sum((F.col("age") < 13).cast("int")).alias("implausible_age_low"),
    F.sum((F.col("age") > 100).cast("int")).alias("implausible_age_high"),
)
sanity.show()

+--------------------+------------------------+-------------------+--------------------+
|returns_before_order|more_returns_than_orders|implausible_age_low|implausible_age_high|
+--------------------+------------------------+-------------------+--------------------+
|                   0|                       0|                  0|                   0|
+--------------------+------------------------+-------------------+--------------------+



## 3. Outlier Detection (IQR method)

We use Tukey's IQR rule (flag values outside `Q1 - 1.5*IQR` / `Q3 + 1.5*IQR`) computed with Spark's
`approxQuantile` (distributed-friendly — avoids pulling data to the driver, which matters at
production scale even though this local sample is small).

**Business call — capping, not deletion:** for a fraud/abuse use case, extreme values are frequently
*the signal itself* (a fraudster requesting an unusually large refund, or an unusually fast
wishlist-to-cart flip suggesting a bot). Deleting them would remove exactly the rows the model most
needs to learn from. We therefore **winsorize (cap) at the 1st/99th percentile** rather than drop
rows, preserving row count and the extremeness of the behaviour while limiting leverage from
data-entry-error-scale outliers.

In [6]:
continuous_cols = [
    "avg_order_value_usd", "refund_amount_requested_usd", "days_to_return",
    "wishlist_to_cart_time_hrs", "account_age_days", "return_rate_pct",
    "previous_dispute_count", "customer_support_contacts",
]

bounds = {}
for c in continuous_cols:
    q1, q3 = bronze_df.approxQuantile(c, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    p01, p99 = bronze_df.approxQuantile(c, [0.01, 0.99], 0.01)
    n_outliers = bronze_df.filter((F.col(c) < lo) | (F.col(c) > hi)).count()
    bounds[c] = {"iqr_lo": lo, "iqr_hi": hi, "cap_lo": p01, "cap_hi": p99, "n_outliers": n_outliers}
    print(f"{c:35s} IQR outliers: {n_outliers:5d} ({n_outliers/bronze_df.count()*100:4.1f}%)  "
          f"| capping to [{p01:.2f}, {p99:.2f}]")

avg_order_value_usd                 IQR outliers:  2955 ( 4.9%)  | capping to [15.00, 799.96]


refund_amount_requested_usd         IQR outliers:  3593 ( 6.0%)  | capping to [12.12, 836.68]


days_to_return                      IQR outliers:   615 ( 1.0%)  | capping to [1.00, 55.00]


wishlist_to_cart_time_hrs           IQR outliers:     0 ( 0.0%)  | capping to [0.10, 72.00]


account_age_days                    IQR outliers:     0 ( 0.0%)  | capping to [1.00, 2500.00]


return_rate_pct                     IQR outliers:   694 ( 1.2%)  | capping to [0.00, 84.70]


previous_dispute_count              IQR outliers:  9017 (15.0%)  | capping to [0.00, 5.00]


customer_support_contacts           IQR outliers: 11916 (19.9%)  | capping to [0.00, 6.00]


In [7]:
silver_df = bronze_df
for c, b in bounds.items():
    silver_df = silver_df.withColumn(
        c, F.when(F.col(c) < b["cap_lo"], b["cap_lo"])
            .when(F.col(c) > b["cap_hi"], b["cap_hi"])
            .otherwise(F.col(c))
    )
print("Outlier capping applied to:", list(bounds.keys()))

Outlier capping applied to: ['avg_order_value_usd', 'refund_amount_requested_usd', 'days_to_return', 'wishlist_to_cart_time_hrs', 'account_age_days', 'return_rate_pct', 'previous_dispute_count', 'customer_support_contacts']


## 4. Feature Engineering (Silver Layer)

Derived features chosen to directly serve the three sub-problems from the business framing:

| Feature | Serves | Rationale |
|---|---|---|
| `return_to_order_ratio` | Return risk | Cross-checks the provided `return_rate_pct`; recomputed independently as a data-quality guard |
| `fraud_signal_score` | Return fraud | Additive count of 6 known fraud red-flags (address change, refund-to-different-account, multiple accounts, invalid tracking, no photo evidence, packaging tampered) — a single interpretable risk score for stakeholders, in addition to being a raw model feature |
| `is_quality_issue_reason` | Product quality | Flags `return_reason` in {Defective/Broken, Quality Issue, Not As Described} — the slice a quality/ops team owns vs. one a fraud team owns |
| `refund_to_order_value_ratio` | Return fraud | Refund requested relative to order value; abusers/fraudsters skew this ratio |
| `account_age_years` | Return risk | More interpretable for stakeholder-facing dashboards than raw days |
| `is_new_customer` | Return risk | New-segment customers behave differently (less history to trust) |
| `high_value_no_evidence` | Return fraud | Interaction flag: high-value item *and* no photo evidence — a specific known abuse pattern

In [8]:
quality_reasons = ["Defective/Broken", "Quality Issue", "Not As Described"]

silver_df = (
    silver_df
    .withColumn(
        "return_to_order_ratio",
        F.when(F.col("total_orders_lifetime") > 0,
               F.col("total_returns_lifetime") / F.col("total_orders_lifetime"))
         .otherwise(0.0)
    )
    .withColumn(
        "fraud_signal_score",
        F.col("address_change_before_delivery")
        + F.col("refund_to_different_account")
        + F.col("multiple_accounts_flag")
        + (1 - F.col("tracking_number_valid"))
        + (1 - F.col("photo_evidence_provided"))
        + (1 - F.col("return_packaging_intact"))
    )
    .withColumn("is_quality_issue_reason", F.col("return_reason").isin(quality_reasons).cast("int"))
    .withColumn(
        "refund_to_order_value_ratio",
        F.when(F.col("avg_order_value_usd") > 0,
               F.col("refund_amount_requested_usd") / F.col("avg_order_value_usd"))
         .otherwise(0.0)
    )
    .withColumn("account_age_years", F.round(F.col("account_age_days") / 365.25, 2))
    .withColumn("is_new_customer", (F.col("customer_segment") == "New").cast("int"))
    .withColumn(
        "high_value_no_evidence",
        ((F.col("is_high_value_item") == 1) & (F.col("photo_evidence_provided") == 0)).cast("int")
    )
)

silver_df.select(
    "order_id", "return_to_order_ratio", "return_rate_pct", "fraud_signal_score",
    "is_quality_issue_reason", "refund_to_order_value_ratio", "account_age_years",
    "is_new_customer", "high_value_no_evidence"
).show(5, truncate=False)

+----------+---------------------+---------------+------------------+-----------------------+---------------------------+-----------------+---------------+----------------------+
|order_id  |return_to_order_ratio|return_rate_pct|fraud_signal_score|is_quality_issue_reason|refund_to_order_value_ratio|account_age_years|is_new_customer|high_value_no_evidence|
+----------+---------------------+---------------+------------------+-----------------------+---------------------------+-----------------+---------------+----------------------+
|ORD2024554|0.0125               |1.2            |0                 |1                      |0.9851219229431541         |4.03             |1              |0                     |
|ORD2019797|0.037037037037037035 |3.7            |2                 |1                      |0.8983231870127096         |3.32             |0              |0                     |
|ORD2058733|0.0                  |0.0            |2                 |0                      |0.9991637047

In [9]:
# Sanity check: engineered fraud_signal_score should separate the classes as expected
silver_df.groupBy("abuse_type").agg(
    F.round(F.avg("fraud_signal_score"), 2).alias("avg_fraud_signal_score"),
    F.round(F.avg("return_to_order_ratio"), 3).alias("avg_return_to_order_ratio"),
    F.round(F.avg("refund_to_order_value_ratio"), 2).alias("avg_refund_to_order_ratio"),
).orderBy(F.desc("avg_fraud_signal_score")).show(truncate=False)

+-----------------+----------------------+-------------------------+-------------------------+
|abuse_type       |avg_fraud_signal_score|avg_return_to_order_ratio|avg_refund_to_order_ratio|
+-----------------+----------------------+-------------------------+-------------------------+
|Fraudulent Return|2.73                  |0.471                    |1.0                      |
|Policy Abuser    |1.81                  |0.612                    |0.95                     |
|Wardrobing       |1.05                  |0.405                    |0.95                     |
|Legitimate       |0.84                  |0.054                    |0.9                      |
+-----------------+----------------------+-------------------------+-------------------------+



**Reading the sanity check:** `fraud_signal_score` and `refund_to_order_value_ratio` should be
visibly higher for *Fraudulent Return* and *Wardrobing* than for *Legitimate* — if not, revisit the
feature definitions before moving to modeling. This cell is the feature-engineering justification
the rubric asks for: it demonstrates *why* each engineered feature earns its place, not just that it
was computed.

## 5. Gold Layer — Encoding, Scaling & Model-Ready Feature Vectors

- **Nominal categoricals** (`country`, `platform`, `device_type`, `payment_method`,
  `product_category`, `return_reason`, `shipping_carrier`) → `StringIndexer` + `OneHotEncoder`
  (no ordinal relationship between categories).
- **`customer_segment`** → encoded via a hand-specified **ordinal** map
  (New < Bronze < Silver < Gold < Platinum) since loyalty tiers *do* have a natural order, which a
  plain one-hot encoding would throw away.
- **Continuous numeric features** → assembled and passed through `StandardScaler`. Tree-based models
  (Random Forest / GBT — our primary classifier) are scale-invariant, but **K-Means clustering**
  (Phase 3, second approach) is distance-based and *requires* scaling, so we standardize once here
  and reuse the same pipeline for both models.
- Binary flag columns (already 0/1) are passed through unscaled.

In [10]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

segment_order = {"New": 0, "Bronze": 1, "Silver": 2, "Gold": 3, "Platinum": 4}
segment_map = F.create_map([F.lit(x) for pair in segment_order.items() for x in pair])
silver_df = silver_df.withColumn("customer_segment_ordinal", segment_map[F.col("customer_segment")].cast("int"))

nominal_cats = ["country", "platform", "device_type", "payment_method",
                "product_category", "return_reason", "shipping_carrier"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in nominal_cats]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in nominal_cats]

binary_flags = [
    "is_high_value_item", "discount_used", "item_returned_opened", "return_packaging_intact",
    "photo_evidence_provided", "tracking_number_valid", "address_change_before_delivery",
    "refund_to_different_account", "multiple_accounts_flag", "review_left_after_return",
    "is_quality_issue_reason", "is_new_customer", "high_value_no_evidence",
]

continuous_features = [
    "age", "account_age_days", "avg_order_value_usd", "refund_amount_requested_usd",
    "days_to_return", "total_orders_lifetime", "total_returns_lifetime", "return_rate_pct",
    "customer_support_contacts", "previous_dispute_count", "wishlist_to_cart_time_hrs",
    "return_to_order_ratio", "fraud_signal_score", "refund_to_order_value_ratio",
    "account_age_years", "customer_segment_ordinal",
]

num_assembler = VectorAssembler(inputCols=continuous_features, outputCol="num_features_raw")
scaler = StandardScaler(inputCol="num_features_raw", outputCol="num_features_scaled", withMean=True, withStd=True)

final_assembler = VectorAssembler(
    inputCols=["num_features_scaled"] + binary_flags + [f"{c}_ohe" for c in nominal_cats],
    outputCol="features"
)

feature_pipeline = Pipeline(stages=indexers + encoders + [num_assembler, scaler, final_assembler])
feature_model = feature_pipeline.fit(silver_df)
gold_df = feature_model.transform(silver_df)

gold_df.select("order_id", "features", "abuse_label", "abuse_type").show(5, truncate=80)

+----------+--------------------------------------------------------------------------------+-----------+-----------------+
|  order_id|                                                                        features|abuse_label|       abuse_type|
+----------+--------------------------------------------------------------------------------+-----------+-----------------+
|ORD2024554|(80,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,19,20,21,26,27,29,39,43,50,58,6...|          0|       Legitimate|
|ORD2019797|(80,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,21,26,32,37,40,47,57,64,75],[1.303...|          0|       Legitimate|
|ORD2058733|(80,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,18,19,21,22,29,39,44,50,51,66,75],...|          0|       Legitimate|
|ORD2015301|(80,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20,21,22,25,31,39,43,47,5...|          2|Fraudulent Return|
|ORD2014206|(80,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,21,26,29,39,41,46,51,64,77],[-1.51...|          0|       Legitimate|
+-------

In [11]:
print(f"Final feature vector length: {gold_df.select('features').first()[0].size}")
gold_df.select("features", "abuse_label").printSchema()

Final feature vector length: 80
root
 |-- features: vector (nullable = true)
 |-- abuse_label: integer (nullable = true)



## 6. Train / Test Split & Persistence

We split 75/25 with a fixed seed for reproducibility. Spark MLlib has no built-in stratified split,
so Phase 3 handles the class imbalance (Legitimate 70% vs. the three abuse classes) via
**class weighting** at the model-training step rather than at split time — splitting by weighted
sampling per class here would leak class-balance assumptions into the "test" set and inflate
evaluation metrics.

Both the engineered Silver table and the encoded Gold feature table are persisted as Parquet so
Phase 3 can load them directly without re-running this pipeline.

In [12]:
train_df, test_df = gold_df.randomSplit([0.75, 0.25], seed=42)
print(f"Train rows: {train_df.count():,}  |  Test rows: {test_df.count():,}")

train_df.groupBy("abuse_type").count().orderBy(F.desc("count")).show()
test_df.groupBy("abuse_type").count().orderBy(F.desc("count")).show()

Train rows: 45,112  |  Test rows: 14,888


+-----------------+-----+
|       abuse_type|count|
+-----------------+-----+
|       Legitimate|31573|
|    Policy Abuser| 5410|
|Fraudulent Return| 4624|
|       Wardrobing| 3505|
+-----------------+-----+



+-----------------+-----+
|       abuse_type|count|
+-----------------+-----+
|       Legitimate|10487|
|    Policy Abuser| 1782|
|Fraudulent Return| 1488|
|       Wardrobing| 1131|
+-----------------+-----+



In [ ]:
import shutil

OUT_DIR = "../data/processed"
shutil.rmtree(OUT_DIR, ignore_errors=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Silver: human-readable engineered features (no vectors) — useful for the report's EDA charts
silver_export_cols = [c for c in silver_df.columns if c not in ("features",)]
silver_df.select(silver_export_cols).write.mode("overwrite").parquet(f"{OUT_DIR}/silver")

# Gold: full encoded/scaled feature table, pre-split, for Phase 3
train_df.write.mode("overwrite").parquet(f"{OUT_DIR}/gold_train")
test_df.write.mode("overwrite").parquet(f"{OUT_DIR}/gold_test")

# Persist the fitted feature pipeline so Phase 3 (or a production scoring job) can
# transform new/incoming data identically without refitting indexers/encoders/scaler
feature_model.write().overwrite().save(f"{OUT_DIR}/feature_pipeline_model")

print("Wrote:", os.listdir(OUT_DIR))

## 7. Handoff to Phase 3 (ML)

**Artifacts produced by this notebook** (in `data/processed/`):
- `silver/` — cleaned, outlier-capped, feature-engineered table in Parquet (for EDA/report charts)
- `gold_train/`, `gold_test/` — encoded + scaled `features` vector column, split 75/25, seed=42
- `feature_pipeline_model/` — the fitted `Pipeline` (indexers, encoders, scaler) for consistent transforms on new data

**Next notebook (`02_ml_modeling.ipynb`) will:**
1. Load `gold_train` / `gold_test`
2. Train a **multi-class classifier** (Random Forest, class-weighted) on `abuse_label` — primary model for return fraud detection
3. Train **K-Means / Bisecting K-Means clustering** on the scaled numeric features — second approach, for product-quality/customer-behavior segmentation
4. Evaluate with precision/recall/F1 per class (accuracy alone is misleading given the 70/12/10/8 imbalance) and silhouette score for the clustering
5. Select and justify the final model(s) for the business recommendation

In [14]:
spark.stop()